# DTW 调参：网格搜索

每个参数给一个候选值列表（`GRID`），把所有组合都试一遍，看哪组结果最好。

- 算出来距离完全相同的组合只算一次：时间惩罚为 0 时 `power`、`dynamics` 不起作用；步惩罚等价于开启、延伸惩罚各加同样的值。
- 每组参数的距离矩阵用徐秋凡的 k-medoids 聚类，跑 `N_SEEDS` 个随机种子取平均，同时按两种标签打分：`true_cluster`（6 类，早晚峰算两类，k=6）和 `shape_cluster`（5 类，早晚峰合并，k=5）。欧氏、皮尔森作参照。
- 排序：ARI 平均 → ARI 最差种子 → NMI → 真标签轮廓系数（距离把真实类型分得多开）→ 开启的参数越少越好 → 越快越好。
- 每算完一组就存进 `results.csv`。中途停掉没关系，重新运行会跳过已经算过的组合。

在 `zrz/dtw` 文件夹里运行，从上到下依次执行。只需要改第一格。

In [ ]:
# 设置：按需要修改
DATA = '../../wpj/predata/数据准备/outputs/simulated/baseline'   # 含 processed/centered.csv 和 ground_truth.csv
OUTPUT = 'outputs/tuning_baseline'                                # 结果保存位置（换网格或数据时建议换新文件夹）
SCALE_FLOOR = 0.1                                                 # 与 dtw.py 默认相同
N_SEEDS = 20                                                      # 每组参数聚类几次取平均
SCHEMES = {'true_cluster': 6, 'shape_cluster': 5}                 # 标签 -> 聚类簇数
WORKERS = 4                                                       # 同时用几个 CPU 核
TOP = 20                                                          # 结果表显示前几名

# 每个参数的候选值：所有组合都会试一遍
GRID = {
    'window':         [0, 0.0834, 0.1667, 0.25],   # 0 = 不限制
    'step_penalty':   [0, 0.1, 0.5, 2],
    'open_penalty':   [0, 0.2, 2],
    'extend_penalty': [0, 0.05, 0.5],
    'time_penalty':   [0, 0.5, 2, 10],
    'power':          [1, 2],                      # 只在 time_penalty > 0 时起作用
    'dynamics':       [0, 0.3, 1],                 # 只在 time_penalty > 0 时起作用
}

In [ ]:
import csv, json, time
from collections import Counter
from multiprocessing import Pool
from pathlib import Path
from IPython.display import Markdown, display

from dtw import read_matrix, prepare_expression, preprocessing_config
from support.tuning import (expand_grid, effective_key, evaluate, rank_key, command, load_results,
                            append_results, parameter_effects)

data, out = Path(DATA), Path(OUTPUT)
ids, times, raw = read_matrix(data/'processed'/'centered.csv')
values = prepare_expression(raw, preprocessing_config(SCALE_FLOOR))
with (data/'ground_truth.csv').open(encoding='utf-8-sig', newline='') as f:
    table = {r['gene_id']: r for r in csv.DictReader(f)}
truth = {s: [table[g][s] for g in ids] for s in SCHEMES}
seeds = tuple(range(1, N_SEEDS + 1))
print(f'{len(ids)} 个基因 × {len(times)} 个时间点')

## 1. 一共要算多少组

In [ ]:
configs, raw_count = expand_grid(GRID)
out.mkdir(parents=True, exist_ok=True)
results = load_results(out/'results.csv')
done = {r['key'] for r in results}
jobs = [(m, {}, values, times, truth, SCHEMES, seeds) for m in ('euclidean', 'pearson') if m not in done]
jobs += [('dtw', q, values, times, truth, SCHEMES, seeds) for q in configs
         if effective_key(q) not in done]

t0 = time.perf_counter(); evaluate(('dtw', configs[0], values, times, truth, SCHEMES, seeds))
one = time.perf_counter() - t0
print(f'网格共 {raw_count} 种组合，去掉等价的后 {len(configs)} 组；已算过 {len(done)} 组，还剩 {len(jobs)} 组。')
print(f'单组约 {one:.1f} 秒，用 {WORKERS} 个核预计约 {one*len(jobs)/max(1, WORKERS)/60:.0f} 分钟'
      '（开窗口的组合更快，实际通常更短）。')

## 2. 运行（可以随时中断，重新运行会接着算）

In [ ]:
started = time.perf_counter()
with Pool(max(1, WORKERS)) as pool:
    for k, rows in enumerate(pool.imap_unordered(evaluate, jobs), 1):
        append_results(out/'results.csv', rows)
        results += rows
        if k % 25 == 0 or k == len(jobs):
            spent = time.perf_counter() - started
            print(f'{k}/{len(jobs)}  已用 {spent/60:.1f} 分钟，预计还要 {spent/k*(len(jobs)-k)/60:.1f} 分钟', flush=True)
print('完成')

## 3. 结果

In [ ]:
def ranking(scheme):
    rows = sorted((r for r in results if r['scheme'] == scheme), key=rank_key)
    for i, r in enumerate(rows, 1):
        r['rank'] = i
    reference = [r for r in rows if r['method'] != 'dtw' or r['name'] == 'standard DTW']
    shown = rows[:TOP] + [r for r in reference if r['rank'] > TOP]
    lines = [f'### 标签 {scheme}（k={SCHEMES[scheme]}），共 {len(rows)} 组', '',
             '| 排名 | 配置 | ARI 平均 | ARI 最差 | NMI | 真标签轮廓 | 早晚峰同簇率 | 秒 |',
             '| ---: | --- | ---: | ---: | ---: | ---: | ---: | ---: |']
    lines += [f"| {r['rank']} | {r['name']} | {r['ari_mean']:.4f} | {r['ari_min']:.4f} | {r['nmi_mean']:.4f} | "
              f"{r['silhouette_true']:.4f} | {r['early_late_cocluster']:.2f} | {r['seconds']:.1f} |" for r in shown]
    best = next(r for r in rows if r['method'] == 'dtw')
    lines += ['', f"推荐：**{best['name']}**  ", f"`{command(best['params'], SCALE_FLOOR)}`", '',
              '每个参数取某个值时，所有含这个值的组合的平均 ARI 和最好 ARI（看哪些值普遍有帮助）：', '',
              '| 参数 | 值 | 组合数 | 平均 ARI | 最好 ARI |', '| --- | ---: | ---: | ---: | ---: |']
    lines += [f"| {e['parameter']} | {e['value']:g} | {e['configs']} | {e['ari_mean']:.4f} | {e['ari_best']:.4f} |"
              for e in parameter_effects(results, scheme, GRID)]
    return '\n'.join(lines) + '\n'

summary = (f'# DTW 网格搜索结果\n\n数据：`{DATA}`；scale_floor={SCALE_FLOOR}；'
           f'聚类：徐秋凡的 k-medoids，{N_SEEDS} 个种子取平均。\n\n网格：`{json.dumps(GRID)}`\n\n'
           '排序：ARI 平均 → ARI 最差种子 → NMI → 真标签轮廓系数 → 开启的参数越少越好 → 越快越好。'
           '前几名之间差距很小时，可能只是聚类的随机波动。\n\n'
           + '\n'.join(ranking(s) for s in SCHEMES))
display(Markdown(summary))

## 4. 保存结果

In [ ]:
(out/'summary.md').write_text(summary, encoding='utf-8')
pick = lambda r: {'name': r['name'], 'params': r['params'], 'command': command(r['params'], SCALE_FLOOR),
                  **{m: r[m] for m in ('ari_mean', 'ari_min', 'nmi_mean', 'silhouette_true')}}
recommended = {'tuned_on': {'data': str(data.resolve()), 'scale_floor': SCALE_FLOOR, 'grid': GRID,
                            'clustering': 'xqf k-medoids', 'clustering_seeds': list(seeds)}, 'schemes': {}}
for scheme, k in SCHEMES.items():
    dtw_rows = sorted((r for r in results if r['scheme'] == scheme and r['method'] == 'dtw'), key=rank_key)
    standard = next((r for r in dtw_rows if r['name'] == 'standard DTW'), None)
    recommended['schemes'][scheme] = {'k': k, 'best': pick(dtw_rows[0]),
                                      'runners_up': [pick(r) for r in dtw_rows[1:6]],
                                      'standard_dtw': pick(standard) if standard else None}
(out/'recommended.json').write_text(json.dumps(recommended, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
print('已保存：', out/'results.csv', out/'summary.md', out/'recommended.json')